In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
import requests
from retry_requests import retry
import datetime

In [46]:
def get_meteo_data (lng: str,
                    lat: str,
					start_date: str,
					end_date: str) -> pd.DataFrame :
    
	print(f"Get: {lat}°N {lng}°E - {start_date}->{end_date}")

	url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
	params = {
		"latitude": lat,
		"longitude": lng,
		"start_date": start_date,
		"end_date": end_date,
		"daily": ["weather_code", "temperature_2m_max", "temperature_2m_min", "apparent_temperature_max", "apparent_temperature_min", "sunrise", "sunset", "daylight_duration", "sunshine_duration", "uv_index_max", "uv_index_clear_sky_max", "rain_sum", "showers_sum", "snowfall_sum", "precipitation_sum", "precipitation_hours", "precipitation_probability_max", "shortwave_radiation_sum", "et0_fao_evapotranspiration", "cloud_cover_mean", "dew_point_2m_mean", "et0_fao_evapotranspiration_sum", "relative_humidity_2m_mean", "snowfall_water_equivalent_sum", "pressure_msl_mean", "surface_pressure_mean", "visibility_mean", "wind_speed_10m_mean", "soil_moisture_0_to_100cm_mean", "soil_temperature_0_to_100cm_mean"],
	}

	response = requests.get(url, params=params, timeout=60)
	response.raise_for_status()
	daily = response.json()["daily"]

	return pd.DataFrame({
		"date": pd.to_datetime(daily["time"]),
		"latitude": lat,
		"longitude": lng,
		"weather_code": daily["weather_code"],
		"temperature_2m_max": daily["temperature_2m_max"],
		"temperature_2m_min": daily["temperature_2m_min"],
		"apparent_temperature_max": daily["apparent_temperature_max"],
		"apparent_temperature_min": daily["apparent_temperature_min"],
		"sunrise": daily["sunrise"],
		"sunset": daily["sunset"],
		"daylight_duration": daily["daylight_duration"],
		"sunshine_duration": daily["sunshine_duration"],
		"uv_index_max": daily["uv_index_max"],
		"uv_index_clear_sky_max": daily["uv_index_clear_sky_max"],
		"rain_sum": daily["rain_sum"],
		"showers_sum": daily["showers_sum"],
		"snowfall_sum": daily["snowfall_sum"],
		"precipitation_sum": daily["precipitation_sum"],
		"precipitation_hours": daily["precipitation_hours"],
		"precipitation_probability_max": daily["precipitation_probability_max"],
		"shortwave_radiation_sum": daily["shortwave_radiation_sum"],
		"et0_fao_evapotranspiration": daily["et0_fao_evapotranspiration"],
		"cloud_cover_mean": daily["cloud_cover_mean"],
		"dew_point_2m_mean": daily["dew_point_2m_mean"],
		"et0_fao_evapotranspiration_sum": daily["et0_fao_evapotranspiration_sum"],
		"relative_humidity_2m_mean": daily["relative_humidity_2m_mean"],
		"snowfall_water_equivalent_sum": daily["snowfall_water_equivalent_sum"],
		"pressure_msl_mean": daily["pressure_msl_mean"],
		"surface_pressure_mean": daily["surface_pressure_mean"],
		"visibility_mean": daily["visibility_mean"],
		"wind_speed_10m_mean": daily["wind_speed_10m_mean"],
		"soil_moisture_0_to_100cm_mean": daily["soil_moisture_0_to_100cm_mean"],
		"soil_temperature_0_to_100cm_mean": daily["soil_temperature_0_to_100cm_mean"],
	})

In [ ]:
import time

failed_calls = [] 

def fetch_meteo_data_year(code_station: str,
                           lng: str,
                           lat: str,
                           start_date: str,
                           end_date: str) -> pd.DataFrame:
    import time

    start   = pd.to_datetime('2016-01-01')
    end     = pd.to_datetime(end_date)
    frame   = []

    for year in range(start.year, end.year + 1):
        year_start = max(start, pd.Timestamp(year=year, month=1, day=1))
        year_end = min(end, pd.Timestamp(year=year, month=12, day=31))
        current_start = year_start.strftime("%Y-%m-%d")
        current_end = year_end.strftime("%Y-%m-%d")

        try:
            df_year = get_meteo_data(
                lng,
                lat,
                current_start,
                current_end
            )
            df_year["code_station"] = code_station

            df_year.to_csv (f"meteo_data_{year_start}_{code_station}.csv")

            frame.append(df_year)
            time.sleep(10)

        except Exception as e:
            print(f"Erreur station {code_station}, année {year}: {e}")
            
            failed_calls.append({
                "code_station": code_station,
                "lng": lng,
                "lat": lat,
                "start_date": current_start,
                "end_date": current_end,
                "error": str(e),
            })

    if not frame:
        return pd.DataFrame()

    return pd.concat(frame, ignore_index=True)

In [50]:
df_piezo = pd.read_csv ("station_data.csv" )

frame=[]
retry_frame = []

# récupération des data par années
for start_date, end_date, lat, lng, code_station in zip(df_piezo["date_debut_mesure"], 
                                                        df_piezo["date_fin_mesure"], 
                                                        df_piezo["latitude"], 
                                                        df_piezo["longitude"], 
                                                        df_piezo["code_station"]):
    frame.append(fetch_meteo_data_year(code_station,
                                       lng,
                                       lat,
                                       start_date,
                                       end_date))

# retraitement des erreurs
for call in failed_calls:
    try:
        df_retry = get_meteo_data(call["lng"], 
                                  call["lat"], 
                                  call["start_date"], 
                                  call["end_date"])
        df_retry["code_station"] = call["code_station"]
        
        df_retry.to_csv (f"meteo_data_{call["start_date"]}_{code_station}.csv")

        retry_frame.append(df_retry)
        time.sleep(10)
    
    except Exception as e:
        print(f"Échec persistant pour {call['code_station']} ({call['start_date']} - {call['end_date']}): {e}")

# concat
df_meteo = pd.concat([frame,retry_frame], ignore_index=True)
df_meteo.to_csv (f"meteo_data_full.csv")
display (df_meteo.head())

Get: 4.811397298°N 43.539904887°E - 2016-01-01->2016-12-31
Erreur station 10192X0095/P21B, année 2016: 429 Client Error: Too Many Requests for url: https://historical-forecast-api.open-meteo.com/v1/forecast?latitude=4.811397298&longitude=43.539904887&start_date=2016-01-01&end_date=2016-12-31&daily=weather_code&daily=temperature_2m_max&daily=temperature_2m_min&daily=apparent_temperature_max&daily=apparent_temperature_min&daily=sunrise&daily=sunset&daily=daylight_duration&daily=sunshine_duration&daily=uv_index_max&daily=uv_index_clear_sky_max&daily=rain_sum&daily=showers_sum&daily=snowfall_sum&daily=precipitation_sum&daily=precipitation_hours&daily=precipitation_probability_max&daily=shortwave_radiation_sum&daily=et0_fao_evapotranspiration&daily=cloud_cover_mean&daily=dew_point_2m_mean&daily=et0_fao_evapotranspiration_sum&daily=relative_humidity_2m_mean&daily=snowfall_water_equivalent_sum&daily=pressure_msl_mean&daily=surface_pressure_mean&daily=visibility_mean&daily=wind_speed_10m_mean&

KeyboardInterrupt: 